In [ ]:
# ! pip install -q openai datasets pandas tqdm dotenv

### Imports

In [1]:
from datasets import load_dataset
from openai import OpenAI
import os
import json
import mlflow
from utils import generate_urls, calculate_invoice_accuracies, key_level_metrics, calculate_individual_invoice_accuracies, convert_base64_to_pil, retrieve_token_usage
from prompt import register_prompt

from dotenv import load_dotenv
from mlflow.entities import Feedback
from mlflow.genai import scorer

load_dotenv()

d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


True

### Config

In [2]:
MLFLOW_TRACKING_URI = "http://localhost:5000/"
MODEL_NAME = "gpt-5-nano"
REASONING = "medium"
MLFLOW_EXPERIMENT_NAME = "cord-v2-gpt5-baseline"
PROMPT_NAME = "invoice-extraction-gpt5-prompt"
PROMPT_VERSION = "1"

### Initialize MLflow and OpenAI environment

In [3]:
client = OpenAI()
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
mlflow.openai.autolog()

### Register prompt and model

In [4]:
register_prompt(prompt_name=PROMPT_NAME)

You are a Vision Language Model designed to extract structured data from invoice receipts.
    Task:
    Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

    Requirements:
    1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
    2. Preserve exact formatting for all the extracted values.  
    3. Do not output fields that lack data—omit empty keys.  
    4. Do not add any information not present in the invoice.
    5. In case of prices and currencies, ensure to maintain the original format without any modifications.

    Schema:
    {{schema}}

    Output:
    Return valid, minimal JSON matching this schema - no extraneous keys or null values.
    
Prompt - invoice-extraction-gpt5-prompt already exists.


### Load the dataset

In [5]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

In [6]:
example_1 = json.loads(dataset["validation"][0]["ground_truth"])['gt_parse']
example_2 = json.loads(dataset["validation"][1]["ground_truth"])['gt_parse']
example_3 = json.loads(dataset["validation"][2]["ground_truth"])['gt_parse']

### Data Preparation

In [7]:
with open("schema.json", "r") as f:
    schema_dict = json.load(f)

schema_dict

{'menu': {'nm': 'name of the menu',
  'num': 'identification number of menu',
  'unitprice': 'unit price of menu',
  'cnt': 'quantity of menu',
  'discountprice': 'discounted price of menu',
  'price': 'total price of menu',
  'itemsubtotal': 'price of each menu after discount applied',
  'vatyn': 'whether the price includes tax or not',
  'etc': 'others',
  'sub': {'nm': 'name of submenu',
   'unitprice': 'unit price of submenu',
   'cnt': 'quantity of submenu',
   'price': 'total price of submenu',
   'etc': 'others'}},
 'sub_total': {'price': 'subtotal price',
  'discount_price': 'discounted price in total',
  'service_price': 'service charge',
  'othersvc_price': 'added charge other than service charge',
  'tax_price': 'tax amount',
  'etc': 'others'},
 'total': {'total_price': 'total price',
  'etc': 'others',
  'cashprice': 'amount of price paid in cash',
  'changeprice': 'amount of change in cash',
  'creditcardprice': 'amount of price paid in credit/debit card',
  'emoneyprice'

In [ ]:
test_dataset = dataset["test"].select(range(5))
url_list, ground_truth_list = generate_urls(dataset=test_dataset)

100%|██████████| 5/5 [00:00<00:00, 39.10it/s]


### Inference and Evaluation

In [10]:
if not os.path.exists("artifacts"):
    os.makedirs("artifacts")
    
eval_dataset = []
for index, url in enumerate(url_list):
    eval_dict = {
        "inputs": {"image_base64": url, "schema": schema_dict},
        "expectations": {"expected_response" : ground_truth_list[index]}
    }
    eval_dataset.append(eval_dict)

eval_dataset

[{'inputs': {'image_base64': '...',
   'schema': {'menu': {'nm': 'name of the menu',
     'num': 'identification number of menu',
     'unitprice': 'unit price of menu',
     'cnt': 'quantity of menu',
     'discountprice': 'discounted price of menu',
     'price': 'total price of menu',
     'itemsubtotal': 'price of each menu after discount applied',
     'vatyn': 'whether the price includes tax or not',
     'etc': 'others',
     'sub': {'nm': 'name of submenu',
      'unitprice': 'unit price of submenu',
      'cnt': 'quantity of submenu',
      'price': 'total price of submenu',
      'etc': 'others'}},
    'sub_total': {'price': 'subtotal price',
     'discount_price': 'discounted price in total',
     'service_price': 'service charge',
     'othersvc_price': 'added charge other than service charge',
     'tax_price': 'tax amount',
     'etc': 'others'},
    'total': {'total_price': 'total price',
     'etc': 'others',
     'cashprice': 'amount of price paid in cash',
     'chang

In [11]:
@scorer
def exact_match(inputs, outputs, expectations, trace) -> Feedback:
    outputs = json.loads(outputs)
    expectations = expectations['expected_response']
    trace_id = trace.info.trace_id

    # Create child run for every invoice
    with mlflow.start_run(parent_run_id=parent_run.info.run_id, nested=True, run_name=trace_id):

        # Log the prediction and ground truth
        mlflow.log_dict(outputs, "prediction.json")
        mlflow.log_dict(expectations, "ground_truth.json")


        # Compute accuracy @ invoice level
        pred_df, acc = calculate_individual_invoice_accuracies(ground_truth=expectations, output=outputs)
        mlflow.log_param("trace_id", trace_id)
        mlflow.log_metric("accuracy", acc)

        # Log predictions as artifacts
        pred_df.to_csv(f"artifacts/predictions_{trace_id}.csv", index=False)
        mlflow.log_artifact(local_path=f"artifacts/predictions_{trace_id}.csv")

        # Save the invoice image
        base64_string = inputs['image_base64']
        image = convert_base64_to_pil(base64_string)
        mlflow.log_image(image, f"input_image_{trace_id}.png")

    return Feedback(
        value=round(acc, 2),
        name="Accuracy"
    )

In [12]:
# @mlflow.trace
def predict_fn(image_base64, schema) -> str:
    system_prompt_template = mlflow.genai.load_prompt(name_or_uri=PROMPT_NAME, version=PROMPT_VERSION)
    if "fewshot" in PROMPT_NAME:
        system_prompt = system_prompt_template.format(schema=schema, example1=example_1, example2=example_2, example3=example_3)
    else:
        system_prompt = system_prompt_template.format(schema=schema)
    # print("<<<<<<<<<< System Prompt: ", system_prompt)
    # print("<<<<<<<<<<< Image base64: ", image_base64[:300])
    # print("<<<<<<<<<<< Schema: ", schema)

    response = client.responses.create(
        model=MODEL_NAME,
        reasoning={
            "effort": REASONING,
        },
        text={
            "verbosity": "low"
        },
        input=[
            {
                    "role": "user",
                    "content": [
                        {"type": "input_text", "text": system_prompt},
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{image_base64}"}
                    ]
            }
        ]
    )
    response_text = response.output[1].content[0].text

    return response_text

In [13]:
parent_run = mlflow.start_run(run_name=f"{MODEL_NAME}-{REASONING}-evaluation")

results = mlflow.genai.evaluate(
    data=eval_dataset,
    scorers=[
        exact_match
    ],
    predict_fn=predict_fn,
)

# Log model details
mlflow.log_params({
        "model_name": MODEL_NAME,
    })
mlflow.log_param("reasoning", REASONING)

mlflow.log_param("prompt", PROMPT_NAME)

# Log total number of input and output tokens - to estimate the overall cost
trace_df  = mlflow.search_traces(run_id=parent_run.info.run_id)
input_tokens, output_tokens = retrieve_token_usage(trace_df)
mlflow.log_metric("total_input_tokens", input_tokens)
mlflow.log_metric("total_output_tokens", output_tokens)

# Log cost estimation
with open("cost.json", "r") as f:
    cost_dict = json.load(f)

total_input_cost = (cost_dict[MODEL_NAME]["input"] * input_tokens) / 10 ** 6
total_output_cost = (cost_dict[MODEL_NAME]["output"] * output_tokens) / 10 ** 6
mlflow.log_metric("estimated_cost", total_input_cost + total_output_cost)

# Log aggregated invoice metrics
response_str_list = trace_df["response"].tolist()
response_list = [json.loads(response_str["output"][1]['content'][0]['text']) for response_str in response_str_list]
calculate_invoice_accuracies(response_list, ground_truth_list)
mlflow.log_artifact("artifacts/invoice_metrics.csv")


# Log aggregated key level metrics
key_level_metrics(response_list, ground_truth_list)
mlflow.log_artifact("artifacts/key_metrics.csv")

# Log total execution time
mlflow.log_metric("total_execution_time", trace_df["execution_duration"].sum() / 1000)

# End the parent run
mlflow.end_run()


2025/08/26 23:21:20 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/08/26 23:21:20 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.


Evaluating:   0%|          | 0/5 [Elapsed: 00:00, Remaining: ?] 

🏃 View run tr-bd32c820b142abccc2b0767091444b96 at: http://localhost:5000/#/experiments/681615081818991010/runs/f6b06797a54a43fa89cad7932a390103
🧪 View experiment at: http://localhost:5000/#/experiments/681615081818991010
🏃 View run tr-df4c8652c973a78a220d43c1c4afd75f at: http://localhost:5000/#/experiments/681615081818991010/runs/90433e503f0840b888f1dc8600c99631
🧪 View experiment at: http://localhost:5000/#/experiments/681615081818991010
🏃 View run tr-8897d9822c8278118d3c33b569c34e6f at: http://localhost:5000/#/experiments/681615081818991010/runs/4e0deef09e1944f99ba3c2d357a12db7
🧪 View experiment at: http://localhost:5000/#/experiments/681615081818991010
🏃 View run tr-2c85028e0f6a9985d4973c7b21a9c263 at: http://localhost:5000/#/experiments/681615081818991010/runs/21978b89839849799c864f003404182b
🧪 View experiment at: http://localhost:5000/#/experiments/681615081818991010
🏃 View run tr-38e3edb411d94a81856097fc4197826a at: http://localhost:5000/#/experiments/681615081818991010/runs/57ef7